# 02 LN/TLM yearly combination

Combines `TLM_total` from notebook 01 with one year of cantonal agricultural land
use data (Landwirtschaftliche Nutzungsflaechen, LN) from geodienste.ch.

Run notebook 01 first, at least once per swissTLM3D release.

**Two things changed in this version, and both affect results.**

1. **Selection is on group code 99, not class code 6.** Class 6 is
   "Flaechen ausserhalb der LN" and contains real ground cover: ponds and ditches
   (904), ruderal areas (905), dry stone walls (906), unpaved paths (907),
   region-specific biodiversity areas (908), home gardens (909). Dropping all of
   class 6 removed those from the output. Group 99 is "Ueberlagernde Flaechen"
   and contains only the four codes that genuinely sit on top of other LN
   polygons: 921, 922, 923, 924.

2. **One classification schema in the output.** The previous version concatenated
   the TLM layers (lowercase `class`, `class_code`, ...) with the LN layers
   (capitalised `Class`, `Class_Code`, ...), so the result held two parallel
   half-empty column families. The LN columns are now renamed onto the shared
   lowercase names before the merge.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt

import tlm_ln
from tlm_ln import config, pipeline

tlm_ln.setup_logging()

In [ ]:
def plot_map(gdf, title, column=None, max_features=5000, figsize=(9, 9)):
    """Quick visual check of a layer.

    Large layers are sampled for the plot only; the data itself is untouched.
    Kept in the notebook rather than in the package because plotting is a
    property of how you are looking at the data, not of the workflow.
    """
    if gdf.empty:
        print(f"{title}: layer is empty, nothing to plot")
        return
    subset = gdf.sample(max_features, random_state=42) if len(gdf) > max_features else gdf
    if len(subset) < len(gdf):
        print(f"plotting {max_features:,} of {len(gdf):,} features")
    fig, ax = plt.subplots(figsize=figsize)
    if column and column in subset.columns:
        subset.plot(ax=ax, column=column, legend=True, linewidth=0.2, markersize=2)
    else:
        subset.plot(ax=ax, linewidth=0.2, markersize=2)
    ax.set_title(title)
    ax.set_axis_off()
    plt.show()

## 2. Configuration

Set the year here. Everything else comes from `config.toml`, including per-year
overrides if a year's inputs live somewhere different.

In [ ]:
YEAR = 2025

cfg = config.load_year("config.toml", YEAR)

print(f"Year                     : {cfg.year}")
print(f"LN inputs                : {len(cfg.ln_inputs)}")
for path in cfg.ln_inputs:
    print(f"  {path}")
print(f"Lookup                   : {cfg.lookup_table}")
print(f"TLM_total                : {cfg.tlm_total_gpkg}:{cfg.tlm_total_layer}")
print(f"Output                   : {cfg.output_gpkg}")
print(f"Overlap group code       : {cfg.overlap_group_code}")
print(f"Erase with selected only : {cfg.erase_with_selected_only}")
print(f"Overlap layer            : {cfg.create_overlap_layer}")

## 3. Run

The whole workflow in one call. On national data the slow steps are reading the
LN shapefiles (several minutes), removing duplicate geometries, and the erase.

The optional pairwise overlap layer is off by default because it is by far the
slowest step. Turn it on in `config.toml` when you need the statistics.

In [ ]:
result = pipeline.run_year(cfg, write=True)

layers = result["layers"]

## 4. Checks

### Land use codes outside their validity window

The lookup records `Gueltig_Von` and `Gueltig_Bis` for around thirty codes. Reis
(509) runs to 2022, Kichererbsen (540) starts in 2023, Ackerschonstreifen appears
as 555 up to 2022 and again as 950 from 2023.

The join does not consult those windows, so a feature carrying a retired or
not-yet-introduced code still gets a class. An empty table below means everything
is consistent. Rows mean something is worth a look: a stale cantonal delivery, a
mislabelled download year, or a lookup that predates the year being processed.

This warns rather than fails, because whether cantonal entry systems restrict
farmers to currently valid codes is not established.

In [ ]:
result["code_validity"]

### Codes flagged as overlaying that the group code does not exclude

In the Feb 2026 lookup, six codes carry `ueberlagernd = 1` but only four sit in
group 99. The other two are 927 "Andere Baeume" and 928 "Andere Elemente", both
region-specific biodiversity areas, both in group 60.

Under the group-99 rule those two are kept, even though the data model says they
overlay something else. Trees on a meadow counted as their own surface means that
area is counted twice.

Whether that is an oversight in the lookup or deliberate is a question for the
lookup's authors, which is why this reports rather than decides.

In [ ]:
result["overlap_flag_mismatch"]

### Gaps left by the selection

`TLM_total` is erased with the LN layer and only part of that layer is merged
back, so the difference becomes a hole. This measures how much.

Overlaying features sit on top of other LN polygons by definition, so most of the
dropped area is covered by something that stays. The part that is not is the real
gap, and isolated trees and avenues are the likely source: an avenue along a road
need not lie inside any other parcel.

If `gap_area_km2` is negligible, leave `erase_with_selected_only = false` and
match the concept diagram. If it is not, setting it true closes the gaps.

In [ ]:
result["gaps"]

In [ ]:
# Feature counts and areas for every layer in the run.
result["areas"]

In [ ]:
# Features carrying no classification in the final layer.
result["completeness"]

In [ ]:
# Area per class in the final layer. Worth comparing against last year's run.
result["by_class"]

## 5. Maps

In [ ]:
plot_map(layers[f"LN_{cfg.year}_sel"], f"LN {cfg.year}: features kept", column="Class_Code")

In [ ]:
dropped = layers[f"LN_{cfg.year}_overlay_dropped"]
plot_map(dropped, f"LN {cfg.year}: overlaying features dropped (group {cfg.overlap_group_code})")

In [ ]:
plot_map(layers[f"TLM_LN_{cfg.year}"], f"Final TLM_LN_{cfg.year}", column="class_code")

## 6. Running several years

The year is a parameter, so a back series is a loop rather than seven edited
copies of this notebook. Each year writes its own GeoPackage, manifest and
reports.

Bear in mind that each year is joined against whichever lookup `config.toml`
names. If you have vintage lookups matching each year, add per-year overrides in
the `[years.<year>]` sections rather than joining every year against the newest
one.

In [ ]:
# for year in range(2019, 2026):
#     year_cfg = config.load_year("config.toml", year)
#     pipeline.run_year(year_cfg, write=True)
#     print(f"finished {year}")